# FIN 4000 — Chapter 14: Bond Prices and Yields
*Pricing a bond, premium vs. discount, and the inverse price-yield relationship*

**ANSWER KEY — fully worked, instructor reference**

This notebook is the Python companion to the Chapter 14 Excel template — same data, same formulas, same numbers. Where the Excel workshop has you build a formula in a cell, this notebook has you build it in a line of Python instead.

## Step 1 — Entry Quiz *(3 minutes, individually)*

1. What's the difference between a bond's coupon rate and its yield to maturity?
2. Why does a bond trade at a premium when its coupon rate is above the market's required yield, and at a discount when below?
3. If interest rates rise, what happens to the price of an existing bond, and why?

---

In [1]:
import numpy as np

## Step 2 — Worked Example *(10 minutes, instructor-led)*

Price this bond: it pays an annual coupon and matures at face value.

In [2]:
face, coupon_rate, n, ytm = 1000, 0.06, 10, 0.08
coupon = face * coupon_rate

In [3]:
pv_coupons = coupon * (1 - (1 + ytm) ** -n) / ytm
pv_face = face / (1 + ytm) ** n
price = pv_coupons + pv_face
print(f'PV coupons=${pv_coupons:.2f}  PV face=${pv_face:.2f}  Price=${price:.2f}')

PV coupons=$402.60  PV face=$463.19  Price=$865.80


*Price = PV of coupons + PV of face value = $402.60 + $463.19 = $865.80. Since the 6% coupon is below the 8% required yield, the bond sells at a discount — below its $1,000 face value.*

## A provided helper — like Excel's RATE(), you don't need to build this yourself

Excel's `RATE()` and `PV()` are both solving the same equation — they're just built-in. Here's a Python equivalent for `RATE()`: a simple **bisection search** that tries yields between 0% and 50% until it finds the one that reproduces the target price. You'll *use* this function below, the same way you'd click on Excel's `RATE()` — you don't need to understand bisection to use it, though it's worth a look if you're curious how `RATE()` works under the hood (see the IRR backgrounder for more on this).

In [4]:
def bond_price(face, coupon_rate, n, ytm):
    coupon = face * coupon_rate
    return coupon * (1 - (1 + ytm) ** -n) / ytm + face / (1 + ytm) ** n

def find_ytm(target_price, face, coupon_rate, n, lo=0.0001, hi=0.50):
    """Bisection search — robust, but needs ~35-40 iterations for 8 decimals of precision."""
    for _ in range(100):
        mid = (lo + hi) / 2
        if bond_price(face, coupon_rate, n, mid) > target_price:
            lo = mid   # price too high -> yield too low -> search higher
        else:
            hi = mid
    return (lo + hi) / 2

## A faster provided helper — Newton-Raphson

Bisection is simple and always works, but it only halves the search range each step — that's why it takes ~35-40 iterations. **Newton-Raphson** converges dramatically faster by using the *slope* of the price-yield curve to jump straight toward the answer instead of just narrowing a range:

$$y_{new} = y_{old} - \dfrac{\text{Price}(y_{old}) - \text{Target}}{\text{Price}'(y_{old})}$$

Since there's no simple closed-form derivative handed to us here, `find_ytm_newton()` estimates the slope numerically — nudge the yield up and down by a tiny amount (`h`) and see how much the price moves. *(A preview: once you reach Chapter 16, you'll see that this slope has a name — it's essentially the bond's Modified Duration. Newton-Raphson and duration are computing the same thing.)*

In [5]:
def find_ytm_newton(target_price, face, coupon_rate, n, guess=0.10, tol=1e-8, max_iter=50):
    y = guess
    h = 1e-6
    for i in range(max_iter):
        f = bond_price(face, coupon_rate, n, y) - target_price
        slope = (bond_price(face, coupon_rate, n, y + h) - bond_price(face, coupon_rate, n, y - h)) / (2 * h)
        y_new = y - f / slope
        if abs(y_new - y) < tol:
            return y_new, i + 1   # converged -- return the answer AND how many steps it took
        y = y_new
    return y, max_iter

## Step 3 — Pair Work *(12 minutes, with a partner)*

**1.** Price a second bond: same $1,000 face value and 10-year maturity, but an 8% coupon and a 6% yield to maturity.

In [6]:
price_B = bond_price(1000, 0.08, 10, 0.06)
print(f'Bond B price = ${price_B:.2f}')

Bond B price = $1147.20


**2.** Compute the current yield (annual coupon ÷ price) for both bonds. For the first (discount) bond, confirm coupon < current yield < YTM. What's the ordering for the second (premium) bond?

In [7]:
current_yield_A = coupon / price
current_yield_B = (1000 * 0.08) / price_B
print(f'Bond A current yield={current_yield_A:.2%}  (coupon {0.06:.0%} < CY < YTM {ytm:.0%})')
print(f'Bond B current yield={current_yield_B:.2%}  (coupon 8% > CY > YTM 6%)')

Bond A current yield=6.93%  (coupon 6% < CY < YTM 8%)
Bond B current yield=6.97%  (coupon 8% > CY > YTM 6%)


**3.** If the discount bond's yield rises from 8% to 9%, reprice it. Which direction did the price move, and does that match the inverse price-yield relationship?

In [8]:
price_A_9pct = bond_price(face, coupon_rate, n, 0.09)
print(f'Bond A at 9% yield = ${price_A_9pct:.2f}  (down from ${price:.2f})')

Bond A at 9% yield = $807.47  (down from $865.80)


## Step 4 — Python Pass *(7 minutes)*

- Bond price for both bonds using Excel's PV() function — it's the pricing formula built in
- Current yield formulas for both bonds
- YTM backed out from price using RATE() — verify it recovers 8% and 6%

In [9]:
recovered_ytm_A = find_ytm(price, face, coupon_rate, n)
recovered_ytm_B = find_ytm(price_B, 1000, 0.08, 10)
newton_ytm_A, newton_iters = find_ytm_newton(price, face, coupon_rate, n)
print(f'Bisection      : YTM={recovered_ytm_A:.6f}  (fixed at 100 iterations internally)')
print(f'Newton-Raphson : YTM={newton_ytm_A:.6f}  (converged in {newton_iters} iterations)')
print(f'\nSame answer, but Newton-Raphson got there in {newton_iters} steps instead of ~35-40.')

Bisection      : YTM=0.080000  (fixed at 100 iterations internally)
Newton-Raphson : YTM=0.080000  (converged in 4 iterations)

Same answer, but Newton-Raphson got there in 4 steps instead of ~35-40.


## Step 5 — Discussion *(3-5 minutes, no calculation)*

> A bond is trading at a price above its face value. What does that tell you about the relationship between its coupon rate and the market's required yield — and would you expect its current yield to be above or below its yield to maturity?